# Quantum Finance: Option Pricing

This notebook demonstrates quantum amplitude estimation applied to European
call option pricing. The workflow:

1. Encode a **log-normal price distribution** into qubit amplitudes
2. Build a **payoff oracle** encoding max(S - K, 0)
3. Use **quantum amplitude estimation** to extract the expected payoff

This is one of the most studied near-term quantum finance applications.

In [ ]:
from qdk_pythonic.domains.finance import (
    LogNormalDistribution,
    EuropeanCallOption,
    QuantumAmplitudeEstimation,
)

## Price Distribution

Stock prices are commonly modeled as log-normal. We discretize the distribution
into 2^n bins, each represented by a computational basis state. The amplitude
of each state encodes the probability of the price falling in that bin.

In [ ]:
dist = LogNormalDistribution(
    mu=0.05,       # expected return
    sigma=0.2,     # volatility
    n_qubits=4,    # 2^4 = 16 price bins
    bounds=(0.5, 2.0),
)

bins = dist.bin_values()
print(f"Price qubits: {dist.n_qubits}")
print(f"Number of bins: {len(bins)}")
print(f"Price range: [{dist.bounds[0]}, {dist.bounds[1]}]")
print(f"Bin midpoints: {[f'{b:.3f}' for b in bins[:6]]}...")

# Build the state preparation circuit
state_prep = dist.to_state_prep()
prep_circuit = state_prep.to_circuit()
print(f"\nState prep gates: {prep_circuit.total_gate_count()}")
print(f"State prep depth: {prep_circuit.depth()}")

## European Call Option

A European call option pays max(S - K, 0) at expiry, where S is the stock
price and K is the strike price. The `EuropeanCallOption` class builds a
payoff oracle that encodes this function into qubit amplitudes.

In [ ]:
option = EuropeanCallOption(strike=1.0, distribution=dist)

# The payoff oracle encodes max(S - K, 0) via controlled rotations
oracle = option.payoff_oracle()
print(f"Payoff oracle qubits: {oracle.qubit_count()}")
print(f"Payoff oracle gates: {oracle.total_gate_count()}")

# Full pricing circuit includes state prep + payoff + estimation register
full_circuit = option.to_circuit(n_estimation_qubits=4)
print(f"\nFull circuit qubits: {full_circuit.qubit_count()}")
print(f"Full circuit gates: {full_circuit.total_gate_count()}")
print(f"Full circuit depth: {full_circuit.depth()}")

## Amplitude Estimation

`QuantumAmplitudeEstimation` wraps a state-preparation circuit and a Grover
oracle into a QPE-based estimation circuit. More estimation qubits give
higher precision but deeper circuits.

In [ ]:
qae = QuantumAmplitudeEstimation(
    state_prep=prep_circuit,
    oracle=oracle,
    n_estimation_qubits=6,
)
qae_circuit = qae.to_circuit()

print(f"QAE estimation qubits: {qae.n_estimation_qubits}")
print(f"QAE total qubits: {qae_circuit.qubit_count()}")
print(f"QAE total gates: {qae_circuit.total_gate_count()}")
print(f"QAE depth: {qae_circuit.depth()}")

## Precision vs. Circuit Size

More estimation qubits yield exponentially better precision (error ~ 1/2^n)
but the circuit depth grows linearly with the number of oracle calls.

In [ ]:
print(f"{'Est. Qubits':>12}  {'Total Qubits':>13}  {'Gates':>8}  {'Depth':>8}")
print("-" * 47)

for n_est in [2, 4, 6, 8]:
    circ = option.to_circuit(n_estimation_qubits=n_est)
    print(f"{n_est:>12}  {circ.qubit_count():>13}  "
          f"{circ.total_gate_count():>8}  {circ.depth():>8}")